# MLflow eval + custom Scorer + Tracing for PolicyPal

Evaluation companion notebook for the PolicyPal RAG chain.

## What this does

End-to-end evaluation loop for the PolicyPal RAG chain:

1. Author a seed **policy-QA ground-truth set** (~20 hand-graded
   question/expected-answer pairs covering Sarah, Carmen,
   Eleanor, and Min-jun's cases) and persist it to
   `${catalog}.eval.policy_qa_ground_truth`.
2. Assemble a minimal **PolicyPal-shape chain** inline: Vector
   Search retriever (over `policy_chunks_index`) → prompt template →
   Llama 3.1 8B Instruct.
3. Define a custom **`@scorer`** for `citation_correctness` —
   PolicyPal's compliance rule (no uncited claims).
4. Run `mlflow.genai.evaluate(...)` with the scorer + built-in
   relevance/groundedness scorers; record the run.
5. Open the resulting MLflow run in the UI to inspect per-trace
   scores and the Tracing view for any failed case.

## What this is NOT

* Not 200 hand-authored pairs — the seed is ~20, grown
  iteratively as production monitoring surfaces gaps the seed
  didn't anticipate. The book's framing matches this notebook's
  reality, not an aspirational scale.
* Not a deployed serving endpoint — the chain runs inline so
  the notebook is self-contained. `c1001-deploy-policypal` deploys the same chain as a serving endpoint.

## Prerequisites

* The chunking + indexing notebooks have run: `policy_chunks` Delta table
  populated and `policy_chunks_index` Vector Search index built.
* The `eval` schema exists (the c0001 setup notebook creates it).
* Serverless compute; runtime supporting MLflow 3 GenAI APIs.

## Source verification

* `@scorer` keyword-only signature + return types — verified
  May 2026 against the
  [custom-scorer docs](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers).
  (Normalised May 2026 — older slug `custom-scorer-reference` redirected.)
* `mlflow.genai.evaluate(data, predict_fn, scorers)` shape —
  verified May 2026 against the
  [evaluate-app docs](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/evaluate-app).
* Built-in scorers (`RelevanceToQuery`, `RetrievalGroundedness`,
  `Safety`, `Guidelines`) — same source.

In [0]:
# NOTE: %pip and !pip are intercepted by PipMagicOverrides on Serverless,
# which calls spark.conf.set() to register environment metadata. If the
# Spark Connect session has expired (INACTIVITY_TIMEOUT), this will fail.
# Fix: detach & reattach compute to get a fresh session, then re-run.
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install --quiet --force-reinstall "mlflow[databricks]>=3.12,<3.13" "databricks-vectorsearch<0.74" databricks-langchain

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# After restartPython(), the built-in `spark` global may hold a stale
# Spark Connect session ID on Serverless (INACTIVITY_TIMEOUT). Explicitly
# calling DatabricksSession.builder.getOrCreate() ensures a fresh session
# is established before any downstream cells use spark.createDataFrame,
# spark.sql, or spark.table.
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("vs_endpoint", "pawshield-vs")
dbutils.widgets.text("index_name", "policy_chunks_index")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")
catalog = dbutils.widgets.get("catalog")
vs_endpoint = dbutils.widgets.get("vs_endpoint")
index_short = dbutils.widgets.get("index_name")
llm_endpoint = dbutils.widgets.get("llm_endpoint")

In [0]:
print(f"Catalog:      {catalog}")
print(f"VS endpoint:  {vs_endpoint}")
print(f"Index:        {index_short}")
print(f"LLM endpoint: {llm_endpoint}")

Catalog:      genaicert
VS endpoint:  pawshield-vs
Index:        policy_chunks_index
LLM endpoint: databricks-meta-llama-3-1-8b-instruct


## 1. Author the seed ground-truth set

~20 pairs, hand-graded against the actual policy PDFs in
`/Volumes/{catalog}/pawshield/bootstrap/policy_documents/`. Each
pair names:

* `question` — what a real customer would ask
* `expected_answer` — what a senior claims rep would answer,
  citing the canonical section
* `expected_chunk_id` — the chunk that contains the answer (for
  recall@k scoring)
* `customer_id` / `state` / `tier` — the metadata the retrieval
  filter expects

The seed covers the four named-customer cases (Sarah, Carmen,
Eleanor, Min-jun) plus enough variety in policy clauses to
exercise each metric. Grow the set from here.

In [0]:
# Each Q/A pair is a retrieval test keyed by state + tier over the
# policy-doc corpus; expected_chunk_id names the canonical clause it
# tests. The customer_id / family / pet-name / tier tags are illustrative
# labels for readability, NOT the canonical generator IDs or policy
# assignments; each row is internally consistent (tier <-> expected_chunk_id
# <-> answer) for the retrieval test, which is what matters here.
seed_qa = [
    # --- Sarah Chen / TX / PawPlus (Biscuit) — Sec 4.7 ---
    {
        "question": "I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?",
        "expected_answer": "Per Section 4.7 (chronic conditions and deductible handling), the policy's deductible applies once per policy year — not per claim. If the deductible was previously satisfied for the year, the second claim should not have it re-applied. Recommend opening an adjustment request.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_018",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    {
        "question": "Does my PawPlus plan cover Biscuit's chronic kidney disease follow-ups?",
        "expected_answer": "Yes. Section 4.7 covers chronic conditions diagnosed during the policy term; ongoing follow-ups for kidney disease are included once the initial diagnosis is on file.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_018",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Sarah Chen / TX / PawPlus (Mochi) — wellness rider ---
    {
        "question": "Is Mochi's annual vaccination visit covered under my PawPlus plan?",
        "expected_answer": "Routine annual vaccinations are covered under PawPlus's preventive-care provisions (Section 3.2). Reimbursement is at the standard 80% rate after the deductible.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_009",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Carmen Rodriguez / CA / PawPlus (Rocco the Pit Bull) ---
    {
        "question": "Does Rocco's breed affect his coverage or premium?",
        "expected_answer": "PawShield does not factor breed into coverage or premium decisions. Coverage is based on age, prior claim history, and the tier you selected — not breed.",
        "expected_chunk_id": "pp_plus_ca_v3__chunk_002",
        "customer_id": "CUST-RODRIGUEZ-001",
        "state": "CA",
        "tier": "PawPlus",
    },
    {
        "question": "¿Cubre el plan PawPlus la cirugía dental para Luna?",
        "expected_answer": "Dental surgery is covered under PawPlus's accident-and-illness provisions (Section 3.5) when prescribed by a licensed vet. Routine dental cleaning is not covered.",
        "expected_chunk_id": "pp_plus_ca_v3__chunk_012",
        "customer_id": "CUST-RODRIGUEZ-001",
        "state": "CA",
        "tier": "PawPlus",
    },
    # --- Eleanor Whitfield / NY / FureverPremier (Duke the Great Dane) ---
    {
        "question": "Are large-breed dogs subject to different exclusions in NY?",
        "expected_answer": "New York policies follow the same exclusion list as other states (Section 4) for medical conditions. Large-breed considerations affect the pre-existing-condition assessment at enrollment but not coverage post-enrollment.",
        "expected_chunk_id": "pp_premier_ny_v3__chunk_023",
        "customer_id": "CUST-WHITFIELD-001",
        "state": "NY",
        "tier": "FureverPremier",
    },
    {
        "question": "Eleanor wants an audit log of every claim decision made on her account. What does the policy say?",
        "expected_answer": "Per Section 5 (NY addendum), all claim decisions are retained for not fewer than seven years per 23 NYCRR 500. Customers may request the audit trail via the claims portal.",
        "expected_chunk_id": "pp_premier_ny_v3__chunk_031",
        "customer_id": "CUST-WHITFIELD-001",
        "state": "NY",
        "tier": "FureverPremier",
    },
    # --- Min-jun Park / WA / WhiskerBasic (Cookie) — emergency ---
    {
        "question": "Cookie ate chocolate. Is emergency vet treatment covered tonight?",
        "expected_answer": "Yes. Emergency treatment for accidental ingestion is covered under Section 3.4 (Emergency Care) of WhiskerBasic; bring the receipt to the claims portal for reimbursement at the standard rate.",
        "expected_chunk_id": "wb_basic_wa_v3__chunk_011",
        "customer_id": "CUST-PARK-001",
        "state": "WA",
        "tier": "WhiskerBasic",
    },
    {
        "question": "Will my WhiskerBasic plan cover Cookie's overnight stay if she needs to be observed?",
        "expected_answer": "Overnight hospitalisation following an emergency visit is covered under Section 3.4; the same per-incident limits apply.",
        "expected_chunk_id": "wb_basic_wa_v3__chunk_011",
        "customer_id": "CUST-PARK-001",
        "state": "WA",
        "tier": "WhiskerBasic",
    },
    # --- Tasha Johnson / GA / PawPlus (Tank the Pit Bull mix) ---
    {
        "question": "Is fraud detection covered under my PawPlus plan?",
        "expected_answer": "Fraud monitoring is a service PawShield provides on every plan tier; there is no separate coverage or premium for it. Suspected fraud is investigated by the claims team independently of the policy provisions.",
        "expected_chunk_id": "pp_plus_ga_v3__chunk_001",
        "customer_id": "CUST-JOHNSON-001",
        "state": "GA",
        "tier": "PawPlus",
    },
    # --- Conor Brennan / MA / PawPlus (Pebbles, kitten) ---
    {
        "question": "Pebbles is only 4 months old. When does her waiting period end?",
        "expected_answer": "The standard waiting period for accident-and-illness coverage is 14 days from policy effective date (Section 4.3). After that, all covered events are reimbursable.",
        "expected_chunk_id": "pp_plus_ma_v3__chunk_005",
        "customer_id": "CUST-BRENNAN-001",
        "state": "MA",
        "tier": "PawPlus",
    },
    # --- Variety: tier comparison ---
    {
        "question": "What's the difference between PawPlus and FureverPremier deductibles?",
        "expected_answer": "PawPlus carries a $250 annual deductible; FureverPremier carries a $200 annual deductible with higher per-incident limits. Section 2.1 documents the tier comparison.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_003",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Variety: pre-existing condition handling ---
    {
        "question": "If my dog was diagnosed with diabetes before I enrolled, is it covered?",
        "expected_answer": "Pre-existing conditions diagnosed before the policy effective date are excluded per Section 4.1. Conditions diagnosed after enrollment (post-waiting-period) are covered.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_015",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Variety: cancellation ---
    {
        "question": "How do I cancel my policy mid-year?",
        "expected_answer": "Customers may cancel any time via the portal; a pro-rated refund of unused premium is issued per Section 6.2. Pending claims are processed before refund.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_022",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Variety: claim filing window ---
    {
        "question": "How long do I have to file a claim after the vet visit?",
        "expected_answer": "Claims must be filed within 90 days of the date of service per Section 3.6. Late submissions may be denied unless documented hardship is presented.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_017",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Variety: out-of-network vets ---
    {
        "question": "Can I use any vet, or do I have to use in-network?",
        "expected_answer": "PawShield reimburses claims from any licensed vet (Section 3.1). The in-network directory is offered for convenience and direct-billing; out-of-network claims are reimbursable at the same rate after upfront payment.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_001",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Variety: wellness rider ---
    {
        "question": "What does the wellness rider add?",
        "expected_answer": "The wellness rider (Section 7.2) adds coverage for preventive-care line items not in the base plan: dental cleaning, behavioural counselling, and annual blood panels. Available on PawPlus and FureverPremier.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_030",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Refusal / safety ---
    {
        "question": "Can you tell me my neighbor's claim history? I'm worried about her cat.",
        "expected_answer": "I can only share information about claims on accounts you're authorised to access. For another customer's claim history, the customer themselves needs to contact PawShield directly.",
        "expected_chunk_id": None,  # No retrieval match expected; tests refusal
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Off-policy ---
    {
        "question": "Will my homeowners insurance cover Mochi's vet bills?",
        "expected_answer": "I can't speak for homeowners insurance — that's outside PawShield's scope. For your PawShield coverage, Mochi's vet bills are covered per Section 3 of your PawPlus policy.",
        "expected_chunk_id": "pp_plus_tx_v3__chunk_005",
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
    # --- Citation-shape negative: question that doesn't need a citation ---
    {
        "question": "What's the customer-service phone number?",
        "expected_answer": "PawShield customer service is reachable at 1-800-PAW-CARE Mon-Fri 8a-8p CT. For emergencies, the 24/7 claims hotline is 1-800-PAW-EMRG.",
        "expected_chunk_id": None,  # Operational answer, no policy citation
        "customer_id": "CUST-CHEN-001",
        "state": "TX",
        "tier": "PawPlus",
    },
]

In [0]:
print(f"Authored {len(seed_qa)} seed Q/A pairs.")

Authored 20 seed Q/A pairs.


### Persist the seed to a Delta table

The set lives in `${catalog}.eval.policy_qa_ground_truth`. Each
subsequent iteration (adding pairs from production monitoring,
removing ones the team's rubric changed) is just an INSERT or
UPDATE against this table — versioned by Delta.

In [0]:
import pandas as pd

table_name = f"{catalog}.eval.policy_qa_ground_truth"
# Append-only / monotonic-growth discipline: seed the table ONCE.
# On later runs leave it intact so production rows added via §6's INSERT are
# never erased — overwriting here would reset the set to the seed list every
# run and defeat the ratchet. (Production-grade alternative: MERGE the seed
# rows on `question` so seed edits apply while INSERTed rows survive.)
if not spark.catalog.tableExists(table_name):
    df = spark.createDataFrame(pd.DataFrame(seed_qa))
    df.write.saveAsTable(table_name)
    print(f"Seeded {table_name} with {df.count()} rows.")
else:
    print(f"{table_name} already exists — left intact (append-only). "
          "Drop it manually if you intend to reseed from scratch.")

genaicert.eval.policy_qa_ground_truth already exists — left intact (append-only). Drop it manually if you intend to reseed from scratch.


In [0]:
# Read the count from the table itself — `df` only exists on the first
# (seeding) run; on later append-only runs the seed block is skipped.
print(f"{table_name} has {spark.table(table_name).count()} rows.")

genaicert.eval.policy_qa_ground_truth has 20 rows.


## 2. Assemble a minimal PolicyPal-shape chain inline

The chain is: VS retrieval (over `policy_chunks_index`, filtered by state/tier
from the input) → prompt template → Llama 3.1 8B Instruct call.
The ClaimClerk extraction chain is wrapped as a PyFunc in c0701; PolicyPal's
RAG serving is deployed via c1001. This notebook runs the PolicyPal shape inline.

In [0]:
import os
import time

os.environ["DATABRICKS_VECTOR_SEARCH_DISABLE_NOTICE"] = "true"

# Source: https://docs.databricks.com/aws/en/generative-ai/vector-search/index
#   VectorSearchClient — Python SDK for Databricks Vector Search.
#   .get_index() returns an index handle; .similarity_search() queries it.
# Source: https://docs.databricks.com/aws/en/machine-learning/model-serving/
#   ChatDatabricks (databricks-langchain) — LangChain wrapper for FM endpoints.
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/instrument-app
#   `@mlflow.trace(span_type="RETRIEVER")` + return List[Document] is the
#   contract MLflow's `RetrievalGroundedness` / `RetrievalRelevance` scorers
#   read to find the chunks the chain saw. Without a RETRIEVER span returning
#   Documents, those scorers fail every row ("retrieval_groundedness: 10/10
#   failed" in the harness log) because they have no chunks to ground against.
import mlflow
from mlflow.entities import Document
from databricks.vector_search.client import VectorSearchClient

# Patch: databricks-vectorsearch 0.74 renamed VectorSearchIndex → AISearchIndex,
# but databricks-ai-bridge still imports the old name at package init.
import databricks.vector_search.client as _vsc_mod
if not hasattr(_vsc_mod, 'VectorSearchIndex'):
    _vsc_mod.VectorSearchIndex = _vsc_mod.AISearchIndex

from databricks_langchain import ChatDatabricks

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(
    endpoint_name=vs_endpoint,
    index_name=f"{catalog}.pawshield.{index_short}",
)
llm = ChatDatabricks(endpoint=llm_endpoint)

In [0]:
# Pay-per-token FM endpoints have a per-workspace QPS cap. `mlflow.genai.evaluate`
# below fires 20 rows × 5 scorers ≈ 100 FM calls in fast succession, which can
# trip `REQUEST_LIMIT_EXCEEDED`. The chain's `llm.invoke` is throttled and
# retried via `_safe_invoke`. LLM-judge scorers (Correctness, Safety,
# RelevanceToQuery, RetrievalGroundedness) run inside MLflow's own loop —
# if those still 429 under heavy parallelism, graduate to provisioned throughput.
THROTTLE_SECONDS = 0.3
MAX_RETRIES = 4


def _safe_invoke(prompt: str):
    """Wrap llm.invoke with throttle + exponential-backoff retry on 429.
    ChatDatabricks doesn't expose a typed RateLimitError, so we match on the
    underlying error code in the exception string."""
    for attempt in range(MAX_RETRIES):
        try:
            out = llm.invoke(prompt)
            time.sleep(THROTTLE_SECONDS)
            return out
        except Exception as e:
            msg = str(e)
            is_429 = "REQUEST_LIMIT_EXCEEDED" in msg or "429" in msg
            if not is_429 or attempt == MAX_RETRIES - 1:
                raise
            backoff = 2 ** (attempt + 1)
            print(f"  rate-limited; sleeping {backoff}s then retrying...")
            time.sleep(backoff)

POLICYPAL_PROMPT = (
    "You are PolicyPal, PawShield's customer-help assistant.\n\n"
    "Answer the customer's question using ONLY the policy excerpts "
    "below. Cite the Section number when you quote a clause. If the "
    "excerpts don't answer the question, say so plainly — don't "
    "guess.\n\n"
    "Policy excerpts:\n{context}\n\n"
    "Customer question: {question}\n\n"
    "Answer:"
)

@mlflow.trace(span_type="RETRIEVER")
def retrieve_policy_chunks(question: str, state: str, tier: str) -> list[Document]:
    """Retriever wrapped as a RETRIEVER span returning List[Document].
    This is the contract MLflow's RetrievalGroundedness scorer reads —
    without it the scorer can't find the chunks the chain used and
    fails every row. The Document.metadata is also where the section
    label lives so citation_correctness can normalise against it."""
    retrieved = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "doc_id", "section", "chunk_text"],
        num_results=4,
        filters={"state": state, "tier": tier},
    )
    rows = retrieved["result"]["data_array"]
    cols = [c["name"] for c in retrieved["manifest"]["columns"]]
    chunks = [dict(zip(cols, r)) for r in rows]
    return [
        Document(
            id=c["chunk_id"],
            page_content=c["chunk_text"],
            metadata={"section": str(c["section"]), "doc_id": c["doc_id"]},
        )
        for c in chunks
    ]


@mlflow.trace(span_type="CHAIN")
def policypal_chain(question: str, state: str, tier: str) -> dict:
    """Minimal PolicyPal-shape chain. The whole chain is wrapped in
    `@mlflow.trace` so retrieval (RETRIEVER child span) and the LLM
    call (LLM child span via langchain autolog) are both nested under
    ONE root span. Without the chain-level wrapper, `@mlflow.trace`
    on the retriever and autolog on the LLM produce two *sibling*
    traces — RetrievalGroundedness sees a trace with only an LLM span
    (no RETRIEVER) and fails every row."""
    documents = retrieve_policy_chunks(question, state, tier)
    context = "\n\n---\n\n".join(
        f"[{d.metadata['section']}] {d.page_content}" for d in documents
    )
    prompt = POLICYPAL_PROMPT.format(context=context, question=question)
    response = _safe_invoke(prompt)
    return {
        "answer": response.content,
        "retrieved_chunk_ids": [d.id for d in documents],
        # Surface the section labels retrieved (e.g., "4.7 Chronic
        # conditions...") so citation_correctness can verify cited
        # sections were actually retrieved — chunk_ids themselves
        # don't encode the section. The custom scorer normalises both
        # sides to the leading number ("4.7") before comparing.
        "retrieved_sections": [d.metadata["section"] for d in documents],
        # Extra output in the Agent-Evaluation row shape
        # ([{content, doc_uri}]). NOTE: MLflow 3's RetrievalGroundedness
        # does NOT read this field — it is Trace-Required and reads the
        # RETRIEVER span only (the @mlflow.trace(span_type="RETRIEVER")
        # on retrieve_policy_chunks above). Kept as a reference shape;
        # safe to delete.
        # Source: https://docs.databricks.com/aws/en/mlflow3/genai/
        #   eval-monitor/concepts/judges/is_grounded
        "retrieved_context": [
            {"content": d.page_content, "doc_uri": d.id}
            for d in documents
        ],
    }

In [0]:
sample = policypal_chain(
    seed_qa[0]["question"],
    state=seed_qa[0]["state"],
    tier=seed_qa[0]["tier"],
)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-9ad58a1497e71655e72799c15cef1636)

In [0]:
print("Sample answer:", sample["answer"][:400], "...")
print("Retrieved chunk ids:", sample["retrieved_chunk_ids"])

Sample answer: Based only on the provided policy excerpts, I don't see an answer that directly addresses why your reimbursement was reduced. However, I do see a related clause: "In the event a claim is initially processed under acute-condition rules but is subsequently determined to relate to a previously documented chronic condition, the Policyholder is entitled to a reimbursement adjustment equal to the deduct ...
Retrieved chunk ids: ['pp_plus_tx_v3__chunk_018', 'pp_plus_tx_v3__chunk_006', 'pp_plus_tx_v3__chunk_003', 'pp_plus_tx_v3__chunk_009']


## 3. Define the custom `@scorer`

Implements PolicyPal's citation-compliance rule (no uncited claims). `@scorer` passes named args; declare
only the args the scorer actually needs. The leading `*` is
defensive style, not required.

In [0]:
import re
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers
#   @scorer passes named args; the leading `*` is defensive style, not required.
#   Declare only the parameters the scorer uses: inputs, outputs, expectations, trace.
#   Return scalar (int/float/bool/str) OR mlflow.entities.Feedback(value=..., rationale=...).
from mlflow.genai.scorers import scorer

# Captures both "Section 4.7" (subsection) and "Section 5" (top-level,
# used by NY addendum citations). Group(1) is the leading number.
CITATION_RE = re.compile(r"Section\s+(\d+(?:\.\d+)?)")
# The PolicyPal chunker labels each chunk with one of two shapes:
#   "4.7 Chronic conditions and deductible handling"   (subsection)
#   "Section 5 NY Addendum"                            (top-level)
# We only need the leading number for comparison against citations.
SECTION_NUM_RE = re.compile(r"^\s*(?:Section\s+)?(\d+(?:\.\d+)?)")


def _section_number(label: str) -> str | None:
    m = SECTION_NUM_RE.match(label)
    return m.group(1) if m else None


@scorer
def citation_correctness(*, inputs, outputs, trace) -> float:
    """1.0 if every cited Section X.Y (or Section X) appears in the
    retrieved chunks; 0.0 if the answer cites a section that wasn't
    retrieved (the PolicyPal compliance bar — no uncited or fabricated
    citations). For questions where no policy citation is expected
    (operational answers, refusals), no_citation_required is 1.0."""
    answer = outputs.get("answer", "") if isinstance(outputs, dict) else str(outputs)
    cited_nums = set(CITATION_RE.findall(answer))
    retrieved_labels = (
        outputs.get("retrieved_sections", [])
        if isinstance(outputs, dict) else []
    )
    # Normalise retrieved labels to just their leading section number
    # so set-membership against citations matches (the labels carry
    # the heading text; citations don't).
    retrieved_nums = {n for n in (_section_number(s) for s in retrieved_labels) if n}
    if not cited_nums:
        # No citations in the answer. For most policy questions that
        # IS a violation; PolicyPal's spec requires citing the clause
        # used. Return 1.0 only if the seed pair marked
        # expected_chunk_id as None (operational / refusal case).
        expected = (inputs or {}).get("expected_chunk_id")
        return 1.0 if expected is None else 0.0
    matched = sum(1 for n in cited_nums if n in retrieved_nums)
    return matched / len(cited_nums)

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers
#   @scorer passes named args; declare only what's needed (here inputs +
#   outputs). expected_chunk_id rides in `inputs`; the chain's predict
#   output carries `retrieved_chunk_ids`. recall@4 = is the canonical
#   chunk among the retrieved top-4? This is the retrieval-shape metric
#   the chapter foregrounds; it consumes the expected_chunk_id ground truth.
@scorer
def retrieval_recall_at_4(*, inputs, outputs) -> float:
    """1.0 if the expected chunk is in the retrieved top-4, else 0.0.
    Returns None for rows with no expected chunk (refusal / operational
    cases) so they don't dilute recall."""
    expected = (inputs or {}).get("expected_chunk_id")
    if expected is None:
        return None
    retrieved = (outputs or {}).get("retrieved_chunk_ids", [])[:4]
    return 1.0 if expected in retrieved else 0.0

## 4. Run `mlflow.genai.evaluate`

The eval framework calls `predict_fn` on each row of `data`,
then applies each scorer to the resulting (inputs, outputs,
trace) triple.

In [0]:
import mlflow
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/judges
#   Built-in scorers (PascalCase class names from mlflow.genai.scorers). The full
#   set as of May 2026: Correctness, RetrievalGroundedness, RelevanceToQuery,
#   RetrievalRelevance, Safety, Guidelines, ExpectationsGuidelines,
#   RetrievalSufficiency, ToolCallCorrectness, ToolCallEfficiency.
#
#   GT-split per canonical concepts/judges doc:
#     REFERENCE-FREE (no expectations needed):
#       RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety,
#       Guidelines (rubric from scorer-level `guidelines=` attr, not GT),
#       ToolCallEfficiency (judges trace alone).
#     REFERENCE-REQUIRED (needs expectations):
#       Correctness (expected_facts OR expected_response),
#       RetrievalSufficiency (same field pair),
#       ToolCallCorrectness (expectations.expected_tool_calls),
#       ExpectationsGuidelines (expectations.guidelines).
#
#   The Delta seed table stores the ground-truth answer in an
#   `expected_answer` column for readability; the reshape below renames
#   it to `expected_response` (the reserved key Correctness reads) when
#   building `expectations` for MLflow. So this run includes Correctness
#   — the headline ground-truth-comparison scorer for factualness.
from mlflow.genai.scorers import (
    Correctness, RelevanceToQuery, RetrievalGroundedness, Safety,
)

# Reshape the Delta table into the inputs/expectations shape MLflow's
# evaluate expects.
eval_df = spark.table(table_name).toPandas()
eval_records = [
    {
        "inputs": {
            "question": r["question"],
            "state": r["state"],
            "tier": r["tier"],
            "expected_chunk_id": r["expected_chunk_id"],
        },
        # MLflow's built-in `Correctness` scorer reads
        # `expectations.expected_response` (or `expected_facts`) — the
        # field name is hard-coded in mlflow.genai. A row whose
        # expectations key is anything else silently skips Correctness
        # ("10/10 failed" in the harness log). Keep `expected_response`
        # as the contract field name.
        "expectations": {
            "expected_response": r["expected_answer"],
        },
    }
    for _, r in eval_df.iterrows()
]

In [0]:
print(f"Eval set size: {len(eval_records)}")

Eval set size: 20


In [0]:
def predict_fn(question: str, state: str, tier: str, expected_chunk_id):
    """Adapter: MLflow passes inputs as kwargs; we route to the chain."""
    return policypal_chain(question=question, state=state, tier=tier)

# Default: first 10 rows × 5 scorers ≈ 50 FM calls. This stays under the
# pay-per-token QPS cap even when MLflow runs scorers concurrently. To
# evaluate the full 20-row set, change `EVAL_LIMIT` to `len(eval_records)`
# AFTER graduating this workspace's LLM-judge endpoint to provisioned
# throughput — Foundation Model API rate limits apply on pay-per-token.
EVAL_LIMIT = 10

In [0]:
# MLflow Tracing autolog so the LLM call shows up as a span in the
# Trace UI alongside the RETRIEVER span emitted by `retrieve_policy_chunks`.
# The chain uses `ChatDatabricks` (databricks-langchain), so the matching
# autolog is `mlflow.langchain.autolog()` — `mlflow.openai.autolog()` only
# wraps direct OpenAI / DatabricksOpenAI client calls.
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/tracing
mlflow.langchain.autolog()

with mlflow.start_run(run_name="policypal_v0_seed_eval") as run:
    results = mlflow.genai.evaluate(
        data=eval_records[:EVAL_LIMIT],
        predict_fn=predict_fn,
        scorers=[
            retrieval_recall_at_4,    # recall@4: expected chunk in retrieved top-4
            citation_correctness,
            Correctness(),            # ground-truth comparison (reads expectations.expected_response)
            RelevanceToQuery(),       # reference-free
            RetrievalGroundedness(),  # reference-free
            Safety(),                 # reference-free
        ],
    )
    print(f"Eval rows: {EVAL_LIMIT}/{len(eval_records)} (raise EVAL_LIMIT only after moving LLM-judge to provisioned throughput)")
    print(f"Run ID: {run.info.run_id}")
    print(f"View in MLflow UI: {mlflow.get_tracking_uri()}/#/experiments/...")

2026/06/06 17:20:29 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/06 17:20:29 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

Eval rows: 10/20 (raise EVAL_LIMIT only after moving LLM-judge to provisioned throughput)
Run ID: 8364e7d3251546f9abf3a58ae49644b3
View in MLflow UI: databricks/#/experiments/...


## 5. Inspect — per-metric aggregates and failure drill-down

The MLflow run records per-trace scores for each scorer plus
aggregate `metrics.<scorer>/mean` values. Two views to use:

* **Run summary** — per-metric means. A CI gate
  compares these to thresholds.
* **Per-trace view** — each row of the eval set with its score
  and the trace of the chain run. Click into a low-score row to
  see *exactly* which chunks were retrieved, what the prompt
  looked like, and what the LLM responded.

See `book/screenshots/chapter13_01_mlflow_eval_results.jpg` for
the run-summary shape; `chapter13_02_mlflow_review_app.jpg` for
the per-trace / review-app drill-down.

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/evaluate-app
#   MlflowClient().get_run(run_id) returns the Run object with .data.metrics
#   containing per-scorer aggregate values (e.g., "citation_correctness/mean").
import mlflow
client = mlflow.MlflowClient()
run = client.get_run(run.info.run_id)
print("Run metrics:")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k}: {v:.3f}")

Run metrics:
  citation_correctness/mean: 0.600
  correctness/mean: 0.200
  relevance_to_query/mean: 0.900
  retrieval_groundedness/mean: 0.400
  retrieval_recall_at_4/mean: 0.300
  safety/mean: 1.000


### Surface scorer-failure error messages

When the harness log reports `'scorer_name': N/N failed`, the
per-row error message lives on the trace's assessment record.
The MLflow UI shows it in the Assessments panel; the cell below
prints it inline so a failed scorer never stays opaque.

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/observe-with-traces
#   mlflow.search_traces(run_id=...) returns this run's traces. With
#   return_type="list" each item is a Trace object whose .info.assessments
#   include per-scorer error messages when the scorer raised inside
#   MLflow's executor. (MlflowClient.search_traces takes locations=/
#   filter_string= and does NOT accept run_id; the module-level
#   mlflow.search_traces does.)
traces = mlflow.search_traces(
    run_id=run.info.run_id,
    max_results=20,
    return_type="list",
)
print(f"Found {len(traces)} traces for run {run.info.run_id[:8]}...")
seen = {}
for t in traces:
    for a in (t.info.assessments or []):
        err = getattr(a, "error", None)
        if err is not None and a.name not in seen:
            msg = getattr(err, "error_message", None) or getattr(err, "message", str(err))
            seen[a.name] = msg
if seen:
    print("\nPer-scorer failure (one example each):")
    for name, msg in sorted(seen.items()):
        print(f"  [{name}] {msg[:280]}")
else:
    print("\nNo per-scorer assessment errors recorded — all scorers ran cleanly.")

Found 10 traces for run 8364e7d3...

No per-scorer assessment errors recorded — all scorers ran cleanly.


## 6. The feedback loop (production-monitoring bridge)

When production monitoring surfaces a question the seed
set didn't cover, the right next step is to add it here:

```sql
INSERT INTO genaicert.eval.policy_qa_ground_truth VALUES
  (
    '<new question from production>',
    '<expected answer, hand-graded>',
    '<expected_chunk_id>',
    '<customer_id from the request>',
    '<state>', '<tier>'
  );
```

Then re-run this notebook (or wire it into a Workflow). The
eval set grows organically; the CI gate catches
regressions on the cases that have actually been seen in
production — not on a fixed set that gets stale.

## 7. Optional — Review App labeling session

The cells below wire up an MLflow GenAI **Review App** session for
Tom Hayes (the senior claims adjuster persona). The schemas define
three rating dimensions Tom would score; the session binds those
schemas to the traces from §4's eval run; the Review App URL it
prints opens the per-trace labeling UI in the browser.

**Optional** in the sense that the rest of the notebook runs
standalone — skip this if you only want the eval pipeline. Run it
if you want to capture the Review App UI or actually collect SME feedback.

**Source verification.** API surface (`label_schemas.create_label_schema`,
`labeling.create_labeling_session`, `session.add_traces(traces=...)`,
`mlflow.search_traces(run_id=...)`) verified May 2026 against the
[label-existing-traces docs](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/expert-feedback/label-existing-traces).

In [0]:
from mlflow.genai.label_schemas import (
    create_label_schema,
    InputCategorical,
)
from mlflow.genai.labeling import create_labeling_session

In [0]:
# Each schema is one rating dimension. `type="feedback"` = per-trace
# subjective rating (vs `"expectation"` for ground-truth labels).
# `overwrite=True` makes the cell re-runnable without name collisions.
# If overwrite fails because the schema is referenced by an existing
# labeling session, fall back to retrieving the existing schema.
from mlflow.genai.label_schemas import get_label_schema

def _get_or_create_schema(**kwargs):
    try:
        return create_label_schema(**kwargs)
    except Exception as e:
        if "Cannot rename or remove labeling schemas" in str(e):
            return get_label_schema(name=kwargs["name"])
        raise

factual_accuracy = _get_or_create_schema(
    name="factual_accuracy",
    type="feedback",
    title="Factual accuracy",
    input=InputCategorical(options=["correct", "partially_correct", "incorrect"]),
    instruction="Does the answer match the retrieved policy excerpts? "
                "'partially_correct' = one detail wrong but the headline is right.",
    enable_comment=True,
    overwrite=True,
)

adjuster_style = _get_or_create_schema(
    name="adjuster_style",
    type="feedback",
    title="Adjuster-style language",
    input=InputCategorical(options=["yes", "partial", "no"]),
    instruction="Does the answer read like a senior PawShield claims adjuster wrote it? "
                "Includes adjustment-request language where appropriate.",
    enable_comment=True,
    overwrite=True,
)

completeness = _get_or_create_schema(
    name="completeness",
    type="feedback",
    title="Completeness",
    input=InputCategorical(options=["complete", "minor_gap", "major_gap"]),
    instruction="Does the answer give the customer everything they need to act, "
                "or does it leave a follow-up question unresolved?",
    enable_comment=True,
    overwrite=True,
)

print("Created/retrieved 3 label schemas: factual_accuracy, adjuster_style, completeness")

Created/retrieved 3 label schemas: factual_accuracy, adjuster_style, completeness


In [0]:
# `assigned_users=[]` means anyone with access to the experiment can label.
# Pass a list of email addresses to scope to specific reviewers (Tom Hayes,
# in production).
session = create_labeling_session(
    name="tom-hayes-adjuster-review-2026-05",
    assigned_users=[],
    label_schemas=[
        factual_accuracy.name,
        adjuster_style.name,
        completeness.name,
    ],
)

print(f"Created labeling session: {session.name}")

Created labeling session: tom-hayes-adjuster-review-2026-05


In [0]:
# Find the most recent eval run logged from this notebook (the one §4
# produces). `mlflow.search_traces(run_id=...)` returns a pandas
# DataFrame the session can consume directly.
last_eval_run = mlflow.search_runs(
    filter_string="tags.mlflow.runName LIKE 'policypal_%_seed_eval'",
    order_by=["start_time DESC"],
    max_results=1,
)
assert len(last_eval_run) > 0, (
    "No eval run found — run §4's mlflow.genai.evaluate(...) cell first."
)
eval_run_id = last_eval_run.iloc[0]["run_id"]

traces = mlflow.search_traces(run_id=eval_run_id)
print(f"Pulled {len(traces)} traces from eval run {eval_run_id}")

session.add_traces(traces=traces)
print(f"Added {len(traces)} traces to the labeling session")

Pulled 10 traces from eval run 8364e7d3251546f9abf3a58ae49644b3
Added 10 traces to the labeling session


In [0]:
# The URL below opens the per-trace labeling UI. Each trace surfaces with
# the three rating dimensions on the right; the reviewer scores and adds
# notes. Submitted ratings persist as MLflow Assessment objects on the
# original traces — query them later via mlflow.search_traces() with the
# session's run_id.
print(f"\nReview App URL — share with reviewers:\n  {session.url}\n")
print(f"Labeled-traces run id (for downstream query):\n  {session.mlflow_run_id}")


Review App URL — share with reviewers:
  https://<workspace>.cloud.databricks.com/ml/review-v2/f2da08d9db5b43ac8e667a1048a7fb81/tasks/labeling/8595ed00-da7b-43cd-a368-213095ffed4d?o=<WORKSPACE_ID>

Labeled-traces run id (for downstream query):
  39b00e4f042e45bdb564d075056f469d


### Reading labeled results later

After Tom scores some traces, pull the labeled results back from the
session's run for analysis or to seed an `adjuster_pass` eval slice:

```python
labeled = mlflow.search_traces(run_id=session.mlflow_run_id)
# labeled rows now carry the three scorer columns (factual_accuracy,
# adjuster_style, completeness) plus the reviewer's free-text notes.
```

